# Vera Decomposer Fine-Tuning

Fine-tune LFM2.5-1.2B on claim decomposition using Unsloth + LoRA.

**Sections:**
1. Setup & Installation
2. Load Model
3. Data Preparation
4. Training
5. Save Model
6. Inference
7. Evaluation (Fine-tuned, Zero-shot, Few-shot)
8. Comparison Table

## 1. Setup & Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.57.3
!pip install --no-deps trl==0.22.2
!pip install sentence-transformers

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = 'drive/MyDrive/vera/decomposition'
TRAIN_FILE = f'{DATA_DIR}/vera_train.jsonl'
VAL_FILE = f'{DATA_DIR}/vera_val.jsonl'
TEST_FILE = f'{DATA_DIR}/vera_test.jsonl'
OUTPUT_DIR = f'{DATA_DIR}/model_output'

Mounted at /content/drive


## 2. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = False,
    load_in_8bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "out_proj", "in_proj", "w1", "w2", "w3"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth: Making `model.base_model.model.model` require gradients


## 3. Data Preparation

In [ ]:
import json
from datasets import Dataset

def load_jsonl(filepath):
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

train_data = load_jsonl(TRAIN_FILE)
val_data = load_jsonl(VAL_FILE)
test_data = load_jsonl(TEST_FILE)

print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")

Train: 1872, Val: 404, Test: 405


In [ ]:
STUDENT_PROMPT = """You are a fact decomposer. Break text into atomic claims with exact quotes.

RULES:
1. Extract all claims - mark verifiable facts as FACTUAL, opinions as OPINION.
2. Each claim must stand alone (replace "he/she/it/they" with actual names).
3. Quote must be EXACT substring from input.
4. Return flat JSON array.

OUTPUT FORMAT:
[
  {
    "quote": "exact text from input",
    "atomic_claim": "Self-contained statement",
    "type": "FACTUAL"
  }
]

If no claims found, return: []"""

TEACHER_PROMPT = """You are an expert fact-checker. Decompose text into atomic, verifiable claims.

RULES:
1. Extract ALL claims - mark as FACTUAL or OPINION
2. Each claim must stand alone (resolve pronouns like "he/she/it")
3. Quote must be EXACT substring from input
4. Return flat JSON array

OUTPUT FORMAT:
[
  {"quote": "...", "atomic_claim": "...", "type": "FACTUAL"}
]

EXAMPLES:

Input: "Albert Einstein developed the theory of relativity. He was born in Germany."
Output:
[
  {"quote": "Albert Einstein developed the theory of relativity", "atomic_claim": "Albert Einstein developed the theory of relativity.", "type": "FACTUAL"},
  {"quote": "He was born in Germany", "atomic_claim": "Albert Einstein was born in Germany.", "type": "FACTUAL"}
]

Input: "I think Python is the best language. It was created by Guido van Rossum in 1991."
Output:
[
  {"quote": "I think Python is the best language", "atomic_claim": "Python is the best programming language.", "type": "OPINION"},
  {"quote": "It was created by Guido van Rossum in 1991", "atomic_claim": "Python was created by Guido van Rossum in 1991.", "type": "FACTUAL"}
]

Now decompose the following:"""

In [ ]:
def format_sample(sample):
    """Convert sample to chat format with type field."""
    input_text = sample['input']
    claims = sample['output'].get('claims', [])

    simplified_claims = [
        {
            "quote": c.get("quote", ""),
            "atomic_claim": c.get("atomic_claim", ""),
            "type": c.get("type", "FACTUAL")
        }
        for c in claims
    ]
    output_json = json.dumps(simplified_claims, ensure_ascii=False)

    conversations = [
        {"role": "system", "content": STUDENT_PROMPT},
        {"role": "user", "content": input_text},
        {"role": "assistant", "content": output_json}
    ]
    return {"conversations": conversations}

train_formatted = [format_sample(s) for s in train_data]
val_formatted = [format_sample(s) for s in val_data]

train_dataset = Dataset.from_list(train_formatted)
val_dataset = Dataset.from_list(val_formatted)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Train: 1872, Val: 404


In [ ]:
def formatting_prompts_func(examples):
    texts = tokenizer.apply_chat_template(
        examples["conversations"],
        tokenize = False,
        add_generation_prompt = False,
    )
    return {"text": [x.removeprefix(tokenizer.bos_token) for x in texts]}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
val_dataset = val_dataset.map(formatting_prompts_func, batched=True)

print("Sample:")
print(train_dataset[0]["text"][:600])

Map:   0%|          | 0/1872 [00:00<?, ? examples/s]

Map:   0%|          | 0/404 [00:00<?, ? examples/s]

Sample:
<|im_start|>system
You are a fact decomposer. Break text into atomic claims with exact quotes.

RULES:
1. Extract all claims - mark verifiable facts as FACTUAL, opinions as OPINION.
2. Each claim must stand alone (replace "he/she/it/they" with actual names).
3. Quote must be EXACT substring from input.
4. Return flat JSON array.

OUTPUT FORMAT:
[
  {
    "quote": "exact text from input",
    "atomic_claim": "Self-contained statement",
    "type": "FACTUAL"
  }
]

If no claims found, return: []<|im_end|>
<|im_start|>user
The Craft is a supernatural romance film.<|im_end|>
<|im_start|>assistant



## 4. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 8,
        per_device_eval_batch_size = 16,
        gradient_accumulation_steps = 1,
        warmup_steps = 50,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "epoch",
        save_strategy = "epoch",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = OUTPUT_DIR,
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1872 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/404 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map (num_proc=16):   0%|          | 0/1872 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/404 [00:00<?, ? examples/s]

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB reserved.")

GPU = NVIDIA A100-SXM4-80GB. Max memory = 79.318 GB.
2.23 GB reserved.


In [ ]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,872 | Num Epochs = 3 | Total steps = 702
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 1 x 1) = 8
 "-____-"     Trainable parameters = 11,108,352 of 1,181,448,960 (0.94% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,0.039700,0.040540
2,0.028200,0.038188
3,0.007200,0.042596


## 5. Save Model

In [ ]:
model.save_pretrained(f"{OUTPUT_DIR}/lora_adapters")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapters")
print(f"Saved to {OUTPUT_DIR}/lora_adapters")

Saved to drive/MyDrive/vera/decomposition/model_output/lora_adapters


## 6. Inference

In [ ]:
FastLanguageModel.for_inference(model)

def decompose_text(text, system_prompt, max_new_tokens=512):
    """Run decomposition with given system prompt."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": text}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        use_cache=True,
        temperature=0.1,
        do_sample=True,
    )

    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

In [ ]:
test_text = "I think the Eiffel Tower is beautiful. It was built in 1889 by Gustave Eiffel."
print("Input:", test_text)
print("\nOutput (fine-tuned):")
print(decompose_text(test_text, STUDENT_PROMPT))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Input: I think the Eiffel Tower is beautiful. It was built in 1889 by Gustave Eiffel.

Output (fine-tuned):
[{"quote": "I think the Eiffel Tower is beautiful", "atomic_claim": "The Eiffel Tower is beautiful.", "type": "OPINION"}, {"quote": "It was built in 1889 by Gustave Eiffel", "atomic_claim": "The Eiffel Tower was built in 1889 by Gustave Eiffel.", "type": "FACTUAL"}]


## 7. Evaluation

Three evaluations:
1. **Fine-tuned** - Our model with student prompt
2. **Zero-shot** - Base model with student prompt (no examples)
3. **Few-shot** - Base model with teacher prompt (has examples)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from tqdm import tqdm

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

EVAL_BATCH_SIZE = 8

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def parse_json_safe(text):
    """Parse JSON from model output. Returns (claims_list, is_valid_json, is_correct_structure)."""
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            has_correct_structure = all(isinstance(c, dict) for c in parsed)
            return parsed, True, has_correct_structure
        elif isinstance(parsed, dict) and 'claims' in parsed:
            claims = parsed['claims']
            has_correct_structure = all(isinstance(c, dict) for c in claims)
            return claims, True, has_correct_structure
        return [], True, False  # Valid JSON but wrong structure
    except:
        try:
            start = text.find('[')
            end = text.rfind(']') + 1
            if start >= 0 and end > start:
                parsed = json.loads(text[start:end])
                has_correct_structure = all(isinstance(c, dict) for c in parsed)
                return parsed, True, has_correct_structure
        except:
            pass
    return [], False, False  # Not valid JSON


def compute_similarity(pred_claims, gt_claims, threshold=0.85):
    """Match claims using embedding similarity - with type safety."""
    if not pred_claims or not gt_claims:
        return 0, 0, 0

    pred_claims = [c for c in pred_claims if isinstance(c, dict)]
    gt_claims = [c for c in gt_claims if isinstance(c, dict)]

    if not pred_claims or not gt_claims:
        return 0, 0, 0

    pred_texts = [c.get('atomic_claim', '') for c in pred_claims]
    gt_texts = [c.get('atomic_claim', '') for c in gt_claims]

    if not any(pred_texts) or not any(gt_texts):
        return 0, 0, 0

    pred_emb = embed_model.encode(pred_texts)
    gt_emb = embed_model.encode(gt_texts)

    sim_matrix = np.dot(pred_emb, gt_emb.T) / (
        np.linalg.norm(pred_emb, axis=1, keepdims=True) *
        np.linalg.norm(gt_emb, axis=1, keepdims=True).T
    )

    matches = 0
    used_gt = set()
    for i in range(len(pred_texts)):
        best_j, best_sim = -1, threshold
        for j in range(len(gt_texts)):
            if j not in used_gt and sim_matrix[i, j] > best_sim:
                best_sim, best_j = sim_matrix[i, j], j
        if best_j >= 0:
            matches += 1
            used_gt.add(best_j)

    p = matches / len(pred_claims) if pred_claims else 0
    r = matches / len(gt_claims) if gt_claims else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    return p, r, f1


def check_quote_accuracy(claims, source_text):
    """Check if quotes appear in source text."""
    if not claims:
        return 0
    claims = [c for c in claims if isinstance(c, dict)]
    if not claims:
        return 0
    correct = sum(1 for c in claims if c.get('quote', '') in source_text)
    return correct / len(claims)

In [ ]:
def batched_generate(texts, system_prompt, curr_model, curr_tokenizer, batch_size=8, max_new_tokens=512):
    """Generate outputs for multiple texts in batches."""
    all_outputs = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]

        batch_messages = [
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": text}
            ]
            for text in batch_texts
        ]

        batch_inputs = curr_tokenizer.apply_chat_template(
            batch_messages,
            tokenize=True,
            add_generation_prompt=True,
            padding=True,
            return_tensors="pt"
        ).to("cuda")

        attention_mask = (batch_inputs != curr_tokenizer.pad_token_id).long()

        outputs = curr_model.generate(
            input_ids=batch_inputs,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            temperature=0.1,
            do_sample=True,
            pad_token_id=curr_tokenizer.pad_token_id,
        )

        for j, output in enumerate(outputs):
            input_len = (batch_inputs[j] != curr_tokenizer.pad_token_id).sum().item()
            decoded = curr_tokenizer.decode(output[input_len:], skip_special_tokens=True)
            all_outputs.append(decoded)

    return all_outputs

In [ ]:
def evaluate_batched(test_data, system_prompt, curr_model, curr_tokenizer, max_samples=100):
    """Evaluate model with batched inference - tracks valid/malformed JSON."""
    samples = test_data[:max_samples]
    input_texts = [s['input'] for s in samples]
    gt_claims_list = [s['output']['claims'] for s in samples]

    print(f"  Generating {len(input_texts)} outputs in batches of {EVAL_BATCH_SIZE}...")
    pred_outputs = batched_generate(input_texts, system_prompt, curr_model, curr_tokenizer, batch_size=EVAL_BATCH_SIZE)

    results = {
        'json_valid': 0,
        'json_correct_structure': 0,
        'json_malformed': 0,
        'precision': [],
        'recall': [],
        'f1': [],
        'quote_accuracy': []
    }

    for pred_output, gt_claims, input_text in zip(pred_outputs, gt_claims_list, input_texts):
        pred_claims, is_valid_json, is_correct_structure = parse_json_safe(pred_output)

        if is_valid_json:
            results['json_valid'] += 1
            if is_correct_structure:
                results['json_correct_structure'] += 1
        else:
            results['json_malformed'] += 1

        p, r, f1 = compute_similarity(pred_claims, gt_claims)
        results['precision'].append(p)
        results['recall'].append(r)
        results['f1'].append(f1)
        results['quote_accuracy'].append(check_quote_accuracy(pred_claims, input_text))

    total = len(samples)
    return {
        'total': total,
        'json_valid': results['json_valid'],
        'json_correct_structure': results['json_correct_structure'],
        'json_malformed': results['json_malformed'],
        'json_valid_pct': results['json_valid'] / total * 100,
        'json_correct_pct': results['json_correct_structure'] / total * 100,
        'precision': np.mean(results['precision']) * 100,
        'recall': np.mean(results['recall']) * 100,
        'f1': np.mean(results['f1']) * 100,
        'quote_accuracy': np.mean(results['quote_accuracy']) * 100,
    }

In [ ]:
print("1. Evaluating FINE-TUNED model (with student prompt)...")
finetuned_results = evaluate_batched(test_data, STUDENT_PROMPT, model, tokenizer, max_samples=100)
print(f"   JSON Valid: {finetuned_results['json_valid']}/{finetuned_results['total']}")
print(f"   JSON Correct Structure: {finetuned_results['json_correct_structure']}/{finetuned_results['total']}")
print(f"   F1: {finetuned_results['f1']:.1f}%")

1. Evaluating FINE-TUNED model (with student prompt)...
  Generating 100 outputs in batches of 8...
   JSON Valid: 100/100
   JSON Correct Structure: 100/100
   F1: 98.7%


In [ ]:
print("\nLoading base model for baselines...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/LFM2.5-1.2B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
)
base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "left"
FastLanguageModel.for_inference(base_model)


Loading base model for baselines...
==((====))==  Unsloth 2026.1.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Lfm2ForCausalLM(
  (model): Lfm2Model(
    (embed_tokens): Embedding(65536, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-1): 2 x Lfm2DecoderLayer(
        (conv): Lfm2ShortConv(
          (conv): Conv1d(2048, 2048, kernel_size=(3,), stride=(1,), padding=(2,), groups=2048, bias=False)
          (in_proj): Linear4bit(in_features=2048, out_features=6144, bias=False)
          (out_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (feed_forward): Lfm2MLP(
          (w1): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (w3): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (w2): Linear4bit(in_features=8192, out_features=2048, bias=False)
        )
        (operator_norm): Lfm2RMSNorm((2048,), eps=1e-05)
        (ffn_norm): Lfm2RMSNorm((2048,), eps=1e-05)
      )
      (2): Lfm2DecoderLayer(
        (self_attn): Lfm2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False

In [ ]:
print("2. Evaluating ZERO-SHOT (base model + student prompt)...")
zeroshot_results = evaluate_batched(test_data, STUDENT_PROMPT, base_model, base_tokenizer, max_samples=100)
print(f"   JSON Valid: {zeroshot_results['json_valid']}/{zeroshot_results['total']}")
print(f"   JSON Correct Structure: {zeroshot_results['json_correct_structure']}/{zeroshot_results['total']}")
print(f"   F1: {zeroshot_results['f1']:.1f}%")

2. Evaluating ZERO-SHOT (base model + student prompt)...
  Generating 100 outputs in batches of 8...
   JSON Valid: 100/100
   JSON Correct Structure: 99/100
   F1: 36.3%


In [ ]:
print("3. Evaluating FEW-SHOT (base model + teacher prompt with examples)...")
fewshot_results = evaluate_batched(test_data, TEACHER_PROMPT, base_model, base_tokenizer, max_samples=100)
print(f"   JSON Valid: {fewshot_results['json_valid']}/{fewshot_results['total']}")
print(f"   JSON Correct Structure: {fewshot_results['json_correct_structure']}/{fewshot_results['total']}")
print(f"   F1: {fewshot_results['f1']:.1f}%")

3. Evaluating FEW-SHOT (base model + teacher prompt with examples)...
  Generating 100 outputs in batches of 8...
   JSON Valid: 100/100
   JSON Correct Structure: 100/100
   F1: 58.1%


## 8. Comparison Table

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame({
    'Metric': [
        'JSON Valid',
        'JSON Correct Structure',
        'JSON Malformed',
        'Precision',
        'Recall',
        'F1',
        'Quote Accuracy'
    ],
    'Zero-shot': [
        f"{zeroshot_results['json_valid']}/{zeroshot_results['total']}",
        f"{zeroshot_results['json_correct_structure']}/{zeroshot_results['total']}",
        f"{zeroshot_results['json_malformed']}/{zeroshot_results['total']}",
        f"{zeroshot_results['precision']:.1f}%",
        f"{zeroshot_results['recall']:.1f}%",
        f"{zeroshot_results['f1']:.1f}%",
        f"{zeroshot_results['quote_accuracy']:.1f}%"
    ],
    'Few-shot': [
        f"{fewshot_results['json_valid']}/{fewshot_results['total']}",
        f"{fewshot_results['json_correct_structure']}/{fewshot_results['total']}",
        f"{fewshot_results['json_malformed']}/{fewshot_results['total']}",
        f"{fewshot_results['precision']:.1f}%",
        f"{fewshot_results['recall']:.1f}%",
        f"{fewshot_results['f1']:.1f}%",
        f"{fewshot_results['quote_accuracy']:.1f}%"
    ],
    'Fine-tuned (ours)': [
        f"{finetuned_results['json_valid']}/{finetuned_results['total']}",
        f"{finetuned_results['json_correct_structure']}/{finetuned_results['total']}",
        f"{finetuned_results['json_malformed']}/{finetuned_results['total']}",
        f"{finetuned_results['precision']:.1f}%",
        f"{finetuned_results['recall']:.1f}%",
        f"{finetuned_results['f1']:.1f}%",
        f"{finetuned_results['quote_accuracy']:.1f}%"
    ]
})

print("\n" + "="*70)
print("COMPARISON RESULTS")
print("="*70)
print(comparison_df.to_markdown(index=False))


COMPARISON RESULTS
| Metric                 | Zero-shot   | Few-shot   | Fine-tuned (ours)   |
|:-----------------------|:------------|:-----------|:--------------------|
| JSON Valid             | 100/100     | 100/100    | 100/100             |
| JSON Correct Structure | 99/100      | 100/100    | 100/100             |
| JSON Malformed         | 0/100       | 0/100      | 0/100               |
| Precision              | 36.5%       | 46.0%      | 99.5%               |
| Recall                 | 36.5%       | 83.0%      | 98.5%               |
| F1                     | 36.3%       | 58.1%      | 98.7%               |
| Quote Accuracy         | 98.0%       | 55.2%      | 100.0%              |


In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

all_results = {
    'zeroshot': zeroshot_results,
    'fewshot': fewshot_results,
    'finetuned': finetuned_results
}

with open(f"{OUTPUT_DIR}/eval_results.json", 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nResults saved to {OUTPUT_DIR}/eval_results.json")


Results saved to drive/MyDrive/vera/decomposition/model_output/eval_results.json


In [3]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"{OUTPUT_DIR}/lora_adapters",
    max_seq_length = 2048,
)

model.push_to_hub_gguf(
    "werstal/vera-decomposer-lfm2.5-1.2b-gguf",
    tokenizer,
    quantization_method = "f16",
    token = None,  # Add hf token here
)

print("Saved GGUF f16 to HuggingFace!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.4: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/tmp/unsloth_gguf_sunzxipm`: 100%|██████████| 1/1 [00:29<00:00, 29.98s/it]


...

Unsloth: Merge process complete. Saved to `/tmp/unsloth_gguf_sunzxipm`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['f16'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['LFM2.5-1.2B-Instruct.F16.gguf']
Unsl

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ....5-1.2B-Instruct.F16.gguf:   1%|          | 23.0MB / 2.34GB            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/werstal/vera-decomposer-lfm2.5-1.2b-gguf
Unsloth: Cleaning up temporary files...
Saved GGUF f16 to HuggingFace!


In [5]:
model.push_to_hub_merged("werstal/vera-decomposer-lfm2.5-1.2b", tokenizer, save_method = "merged_16bit", token = None) # Add hf token here

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `werstal/vera-decomposer-lfm2.5-1.2b`: 100%|██████████| 1/1 [00:29<00:00, 29.89s/it]


Successfully copied all 1 files from cache to `werstal/vera-decomposer-lfm2.5-1.2b`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ....5-1.2b/model.safetensors:   2%|2         | 50.3MB / 2.34GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:07<00:00, 67.00s/it]


Unsloth: Merge process complete. Saved to `/content/werstal/vera-decomposer-lfm2.5-1.2b`


In [6]:
from google.colab import runtime
runtime.unassign()